In [1]:
using DifferentialEquations, LinearAlgebra, Plots, LaTeXStrings, PGFPlotsX

# default(
#     fontfamily = "Computer Modern",    # matches LaTeX default body font
#     guidefont  = font(11),             # axis titles
#     tickfont   = font(10),             # tick labels
#     legendfont = font(10)
# )  

pgfplotsx()

Plots.PGFPlotsXBackend()

In [2]:
# Basis states
r = [0; 1]  # Assume |1>
g = [1; 0]  # Assume |0>

#Initial state
α = 1
β = sqrt(1-α^2)
ψ_a = α.*r + β.*g
ψ_1 = g

ρ_0 = complex(kron(ψ_a,ψ_1) * kron(ψ_a,ψ_1)');

In [3]:
struct parameters
    Ω::Float64     
    γ_Decay::Float64     
    γ_dephase::Float64
    V_nn::Float64     
 end

In [4]:
Δ0 = 3000
δ = -500
function (p::parameters)(t)
    global Δ0 = Δ0
    global δ =  δ
    return (Δ0+δ*t,p.Ω, p.γ_Decay, p.γ_dephase, p.V_nn)
end

p = parameters(20,0,0,-2000)

parameters(20.0, 0.0, 0.0, -2000.0)

In [5]:
p(1)

(2500, 20.0, 0.0, 0.0, -2000.0)

In [6]:
function master_eqn(dρ,ρ,p,t)

    #2x2 Matrices
    σ_x = [0 1; 1 0]
    n = [0 0; 0 1]
    I = [1 0; 0 1]
    σ_minus = [0 1; 0 0]
    σ_plus = [0 0; 1 0]
    σ_z = [1 0; 0 -1]

    #parameters
    Δ_t, Ω, γ_Decay, γ_dephase, V_nn = p(t)
    
    #Hamiltonian
    H = Ω/2 .* kron(I,σ_x) + Δ_t .* kron(I,n) + V_nn .* kron(n,n)
    l_decay = γ_Decay .* (kron(I,σ_minus) * ρ * kron(I,σ_plus) - 0.5.*((kron(I,σ_plus) * kron(I,σ_minus) * ρ) + (ρ * kron(I,σ_plus) * kron(I,σ_minus))))
    l_dephase = γ_dephase .* (kron(I,σ_z) * ρ * kron(I,σ_z) - ρ)

    dρ .= -1im .* (H*ρ - ρ*H) + l_decay + l_dephase

end

master_eqn (generic function with 1 method)

In [7]:
-2*(Δ0+p(0)[end])/δ

4.0

In [8]:
# Time span and solve the problem
tspan = (0.0,-2*(Δ0+p(0)[end])/δ);
eqn = ODEProblem(master_eqn, ρ_0, tspan, p);
sol = solve(eqn, Rodas5(autodiff=false),saveat=0.01, progress=true);

In [9]:
# Extracting populations from the solution
n = [0 0; 0 1]
I = [1 0; 0 1]

rydberg_populations = [tr(kron(I,n)*ρ) for ρ in sol.u]

# Plotting the Rydberg state population over time
plot(sol.t, real(rydberg_populations), label=L"Atom 1", xlabel=L"Time (t)", ylabel=L"Population in $|r\rangle$ state", ylim=(0,1))
hline!([1],color=:red,linestyle=:dash,label=L"limit")
savefig("/Users/akashmalemath/Documents/master_work/rydberg_qec/images/part3/rap/1atom_population_speed2_plot.pdf")

"/Users/akashmalemath/Documents/master_work/rydberg_qec/images/part3/rap/1atom_population_speed2_plot.pdf"

In [15]:
ψ_ideal = α*kron(r,r) + β*kron(g,g)

fidelities = []
time = []
for (t,ρ) in zip(sol.t,sol.u)
    fidelity = ψ_ideal' * ρ * ψ_ideal
    push!(fidelities,fidelity)
    push!(time,t)
end

plot(time,real(fidelities),label="Fidelity of our system",xaxis="Time",yaxis="Fidelity",ylim=(0,1))
hline!([1], color=:red, linestyle=:dash, label="Ideal Fidelity")
savefig("/Users/akashmalemath/Documents/master_work/rydberg_qec/images/part3/rap/1atom_speed2_fidelity_plot.pdf")

"/Users/akashmalemath/Documents/master_work/rydberg_qec/images/part3/rap/1atom_speed2_fidelity_plot.pdf"

In [11]:
ΔT = []
for t in range(0,-2*(Δ0+p(0)[end])/δ)
    push!(ΔT,p(t)[1])
end
plot(time,ΔT,label="detuning",xaxis="Time",yaxis="Δ",title="Detuning")

## pLOTS

In [12]:
# decay = collect(range(0.0,stop=1.0,step=0.1))
# dephase = collect(range(0.0,stop=1.0,step=0.1))

# fidelity = Array{Float64}(undef, length(decay), length(dephase))
# for (i,a) in enumerate(decay)
#     for (j,b) in enumerate(dephase)

#         p = parameters(20,a,b,-2000)
#         tspan = (0.0,-2*(Δ0+p(0)[end])/δ)
#         eqn = ODEProblem(master_eqn, ρ_0, tspan, p)
#         sol = solve(eqn, Rodas5(autodiff=false),saveat=0.01, progress=true)
        
#         ψ_ideal = α*kron(r,r) + β*kron(g,g)
#         f = tr(ψ_ideal' * sol[end] * ψ_ideal)
        
#         fidelity[i,j] = real(f) 
        
#     end
# end

In [13]:
# contourf(dephase,decay,fidelity,levels=20,color=:turbo)
# title!("F(decay,dephase)")
# xlabel!("dephase")
# ylabel!("decay")

In [14]:
# heatmap(dephase, decay, fidelity,
#         xlabel="Dephase (δ)",
#         ylabel="Decay (γ)",
#         title="Fidelity(Decay and Dephase)",
#         color=:viridis)